
DIAGRAMA UML - Estructura de la Base de Datos


┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│     AUTORES     │    │  LIBROS_AUTORES  │    │     LIBROS      │
├─────────────────┤    │   (Tabla Pivot)  │    ├─────────────────┤
│ id (PK)         │◄───┤ autor_id (FK)    │───►│ id (PK)         │
│ nombre          │    │ libro_id (FK)    │    │ titulo          │
└─────────────────┘    └──────────────────┘    │ precio          │
                                               │ rating          │
┌─────────────────┐                            │ categoria_id(FK)│
│   CATEGORIAS    │◄───────────────────────────│ en_stock        │
├─────────────────┤                            │ url             │
│ id (PK)         │                            └─────────────────┘
│ nombre          │
└─────────────────┘

Relaciones:
- AUTORES ↔ LIBROS: Muchos a Muchos (un autor puede escribir varios libros, 
  un libro puede tener varios autores)
- CATEGORIAS ↔ LIBROS: Uno a Muchos (una categoría tiene muchos libros)
""")

In [ ]:
pip install psycopg2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Celda 1: Conexion y creacion de tablas

In [ ]:
# Importamos la herramienta para hablar con PostgreSQL
import psycopg2

# 1. LA CONEXIÓN: Todo estrictamente en minúsculas para que coincida con pgAdmin
conexion = psycopg2.connect(
    host="localhost",
    port="5432",
    database="libros",
    user="postgres",
    password="sebas123" 
)

# 2. EL CURSOR: Es el "mensajero" que va a llevar nuestras órdenes SQL a la base de datos
cursor = conexion.cursor()

# 3. CREACIÓN DE TABLA AUTORES
# execute es ejecutar 
cursor.execute("""
    CREATE TABLE IF NOT EXISTS autores (
        id SERIAL PRIMARY KEY,
        nombre VARCHAR(100) UNIQUE NOT NULL
    )
""")

# Creacion de tabla Categorias
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categorias (
        id SERIAL PRIMARY KEY,
        nombre VARCHAR(100) UNIQUE NOT NULL
    )
""")

# 5. CREACIÓN DE TABLA LIBROS
# DECIMAL(10,2) significa que el precio puede tener hasta 10 números, 2 de ellos decimales
# CHECK asegura que nadie meta un rating de 6 estrellas si el máximo es 5
# FOREIGN KEY conecta el libro con la tabla de categorías
cursor.execute("""
    CREATE TABLE IF NOT EXISTS libros (
        id SERIAL PRIMARY KEY,
        titulo VARCHAR(500) NOT NULL,
        precio DECIMAL(10,2),
        rating INTEGER CHECK (rating >= 0 AND rating <= 5),
        en_stock VARCHAR(100),
        url VARCHAR(1000),
        categoria_id INTEGER,
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    )
""")

# 6. CREACIÓN DE TABLA PUENTE (Libros_Autores)
# Esto maneja la relación "Muchos a Muchos". Solo guarda números de ID.
# ON DELETE CASCADE significa que si borrás un libro, se borra su registro en esta tabla automáticamente.
cursor.execute("""
    CREATE TABLE IF NOT EXISTS libros_autores (
        id SERIAL PRIMARY KEY,
        libro_id INTEGER,
        autor_id INTEGER,
        FOREIGN KEY (libro_id) REFERENCES libros(id) ON DELETE CASCADE,
        FOREIGN KEY (autor_id) REFERENCES autores(id) ON DELETE CASCADE,
        UNIQUE(libro_id, autor_id)
    )
""")

# 7. GUARDAR Y CERRAR
# commit() es como apretar "Guardar" en Word. Si no lo ponés, los cambios se pierden.
conexion.commit()

print("Bloque 1 Listo: Las 4 tablas fueron creadas en PostgreSQL.")

Bloque 1 Listo: Las 4 tablas fueron creadas en PostgreSQL.


# Celda 2: Extraccion de datos de la web



In [2]:
pip install requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
from bs4 import BeautifulSoup
import random
from concurrent.futures import ThreadPoolExecutor
import threading

# 1. PREPARATIVOS
autores_equipo = ["Lucas Fleitas", "Nahuel Aquino", "Marcelo Paniagua", "Cesar Espinola", "Sebasthian Lopez"]
conversor_estrellas = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
sesion = requests.Session() # uso el session porque es una manera mucho mas rapida que get.

todos_los_libros = []
# Creamos un "candado" para que los hilos no se pisen al guardar los datos
bloqueo = threading.Lock()  # dice a la base de datos que si el autor o la categoría ya están registrados
# es el semaforo evita que los hilos choquen y se rompan
print("Iniciando extracción con HILOS")

# 2. DEFINIMOS LA FUNCIÓN QUE RASTREA UNA SOLA PÁGINA
def scrapear_pagina(numero_pagina):
    url_pagina = f"https://books.toscrape.com/catalogue/page-{numero_pagina}.html" # se arma el link exacto 
    respuesta = sesion.get(url_pagina) # se va a la pagina y descarga todo el codigo oculto
    respuesta.encoding = "utf-8"
    soup = BeautifulSoup(respuesta.text, "html.parser") # convierte ese texto desordenado en un formato estructurado para buscar
    
    articulos = soup.find_all("article", class_="product_pod") # busca y retorna los datoss que tienen la info de los libros
    libros_de_esta_pagina = [] # Lista temporal solo para esta página
    
    # 3. EXTRAER DATOS DE LOS 20 LIBROS DE ESTA PÁGINA
    for articulo in articulos:
        # Saca el título completo del atributo 'title' y borra espacios sobrantes.
        titulo = articulo.h3.a["title"].strip()
        # Agarra la segunda palabra de la clase CSS que indica las estrellas.
        rating_clase = articulo.find("p", class_="star-rating")["class"][1]
        # traduce esa palabra usando el diccionarip
        rating = conversor_estrellas.get(rating_clase, 0)
        
        # extrae el texto del precio con todo 
        precio_texto = articulo.find("p", class_="price_color").text
        # borra los simbolos de moneda y cpnvierte a texto decimal 
        precio = float(precio_texto.replace("£", "").replace("Â", "").strip()) 
        # captura el texto que dice si el libro esta disponible en stock 
        stock = articulo.find("p", class_="instock availability").text.strip()
        # arma el link directo del libro pegando la ruta con el enlace del boton 
        url_libro = "https://books.toscrape.com/catalogue/" + articulo.h3.a["href"]
        
        # Entrar al libro para sacar la categoría
        respuesta_libro = sesion.get(url_libro)
        # evita que se rompan los caracteres especiales
        respuesta_libro.encoding = "utf-8"
        # transforma el html de esta nueva pagina en un arbol para buscar datos
        soup_libro = BeautifulSoup(respuesta_libro.text, "html.parser")
        # saca el nombre de la categoria desde el menu 
        categoria = soup_libro.find("ul", class_="breadcrumb").find_all("li")[2].text.strip()
        # guarda todos los datos limpios agrupados dentro de la lista temporal
        libros_de_esta_pagina.append({
            "titulo": titulo, "precio": precio, "rating": rating, 
            "categoria": categoria, "stock": stock, "url": url_libro,
            "autor": random.choice(autores_equipo)
        })
        
    # 4. GUARDADO SEGURO
    # Usamos el candado (Lock) para agregar los 20 libros a la lista general
    # Esto asegura que la lista 'todos_los_libros' no se corrompa si 2 hilos terminan a la vez
    with bloqueo:
        todos_los_libros.extend(libros_de_esta_pagina)
        print(f"Página {numero_pagina}/50 completada por un hilo.")

# 5. EL GESTOR DE HILOS (ThreadPoolExecutor)
# Le decimos que trabaje con 5 obreros (hilos) al mismo tiempo
with ThreadPoolExecutor(max_workers=5) as ejecutor: # me permite ejecutar tareas en paralelo
    # map reparte las 50 páginas entre los 5 hilos automáticamente
    ejecutor.map(scrapear_pagina, range(1, 51))

print(f"\n¡Extracción Turbo terminada! Total de libros recolectados: {len(todos_los_libros)}")

Iniciando extracción con HILOS
Página 2/50 completada por un hilo.
Página 4/50 completada por un hilo.
Página 1/50 completada por un hilo.
Página 5/50 completada por un hilo.
Página 3/50 completada por un hilo.
Página 8/50 completada por un hilo.
Página 9/50 completada por un hilo.
Página 10/50 completada por un hilo.
Página 7/50 completada por un hilo.
Página 6/50 completada por un hilo.
Página 11/50 completada por un hilo.
Página 13/50 completada por un hilo.
Página 12/50 completada por un hilo.
Página 14/50 completada por un hilo.
Página 15/50 completada por un hilo.
Página 16/50 completada por un hilo.
Página 17/50 completada por un hilo.
Página 18/50 completada por un hilo.
Página 19/50 completada por un hilo.
Página 20/50 completada por un hilo.
Página 21/50 completada por un hilo.
Página 22/50 completada por un hilo.
Página 25/50 completada por un hilo.
Página 24/50 completada por un hilo.
Página 23/50 completada por un hilo.
Página 26/50 completada por un hilo.
Página 27/50 com

# Celda 3: Insercion de los datos en las tablas

In [ ]:
import psycopg2

# 1. RECONEXIÓN
# Volvemos a conectarnos por seguridad, por si la conexión del Bloque 1 se cerró
conexion = psycopg2.connect(
    host="localhost",
    port="5432",
    database="libros",
    user="postgres",
    password="sebas123" # Tu contraseña real
)
cursor = conexion.cursor()

print("Iniciando la carga de datos en PostgreSQL..")

# 2. EL BUCLE DE INSERCIÓN
# Vamos a recorrer uno por uno los 1000 libros que scrapeamos
for libro in todos_los_libros:
    
    # --- A. CATEGORÍAS ---
    # ON CONFLICT DO NOTHING: Si la categoría (ej: 'Poetry') ya existe, no hace nada ni tira error.
    cursor.execute("""
        INSERT INTO categorias (nombre) VALUES (%s)
        ON CONFLICT (nombre) DO NOTHING;
    """, (libro["categoria"],))
    
    # Buscamos qué ID le tocó a esa categoría (ya sea nueva o vieja) para dárselo al libro
    cursor.execute("SELECT id FROM categorias WHERE nombre = %s", (libro["categoria"],))
    categoria_id = cursor.fetchone()[0]

    # --- B. AUTORES ---
    # Misma lógica: si "Lucas Fleitas" ya fue guardado antes, lo saltea.
    cursor.execute("""
        INSERT INTO autores (nombre) VALUES (%s)
        ON CONFLICT (nombre) DO NOTHING;
    """, (libro["autor"],))
    
    cursor.execute("SELECT id FROM autores WHERE nombre = %s", (libro["autor"],))
    autor_id = cursor.fetchone()[0]

    # --- C. LIBROS ---
    # Insertamos todos los datos del libro.
    # RETURNING id: Le pedimos a PostgreSQL que nos devuelva el ID del libro que acaba de crear.
    cursor.execute("""
        INSERT INTO libros (titulo, precio, rating, categoria_id, en_stock, url)
        VALUES (%s, %s, %s, %s, %s, %s) RETURNING id;
    """, (libro["titulo"], libro["precio"], libro["rating"], categoria_id, libro["stock"], libro["url"]))
    libro_id = cursor.fetchone()[0]

    # --- D. TABLA INTERMEDIA (Libros_Autores) ---
    # Unimos el ID del libro con el ID del autor para crear la relación
    cursor.execute("""
        INSERT INTO libros_autores (libro_id, autor_id) VALUES (%s, %s)
        ON CONFLICT (libro_id, autor_id) DO NOTHING;
    """, (libro_id, autor_id))

# 3. GUARDAR Y CERRAR
# commit() confirma que queremos guardar todo permanentemente
conexion.commit()

# Cerramos las puertas para ser prolijos y no dejar conexiones "fantasma" consumiendo memoria
cursor.close()
conexion.close()

print("¡Bloque 3 Listo! Los 1000 libros ya están guardados de forma segura en tu base de datos.")

Iniciando la carga de datos en PostgreSQL... (Esto puede tardar unos segundos)
¡Bloque 3 Listo! Los 1000 libros ya están guardados de forma segura en tu base de datos.


# Celda 4: Consultas en SQL

In [10]:
import psycopg2

# 1. CONEXIÓN FINAL
conexion = psycopg2.connect(
    host="localhost",
    port="5432",
    database="libros",
    user="postgres",
    password="sebas123" # Tu contraseña
)
cursor = conexion.cursor()

# Función auxiliar para imprimir las tablas prolijas y no repetir código
def imprimir_resultados(titulo_consulta, query):
    print(f"\n{titulo_consulta}")
    print("-" * 50)
    cursor.execute(query)
    resultados = cursor.fetchall()
    for fila in resultados:
        print(fila)

# ==========================================
# PARTE 1: LAS CONSULTAS "LENTAS" (Sin Índice)
# ==========================================
# Como la columna 'precio' no tiene índice, PostgreSQL tiene que hacer un "Seq Scan" 
# (revisar la tabla fila por fila de principio a fin) para encontrar los baratos.
imprimir_resultados(
    "CONSULTA 1 LENTA: Top 5 libros más baratos",
    """
    SELECT titulo, precio 
    FROM libros 
    WHERE precio < 15.00 
    ORDER BY precio ASC 
    LIMIT 5;
    """
)

# ==========================================
# PARTE 2: CREACIÓN DE ÍNDICES (La Optimización)
# ==========================================
print("\n[⏳ Creando Índices en la base de datos...]")
# IF NOT EXISTS evita errores si corrés esta celda dos veces
# Aceleramos la búsqueda por categoría
cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_categoria ON libros(categoria_id);")
# Aceleramos la búsqueda por nombre exacto de autor
cursor.execute("CREATE INDEX IF NOT EXISTS idx_autores_nombre ON autores(nombre);")
# Aceleramos los cruces (JOINs) en la tabla puente
cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_autores_autor ON libros_autores(autor_id);")

conexion.commit() # Guardamos los índices
print("[✅ Índices creados exitosamente]")

# ==========================================
# PARTE 3: LAS CONSULTAS RÁPIDAS (Con Índice y JOINs)
# ==========================================
# Ahora el motor usa un "Index Scan". Va directo al índice de categorías 
# y salta a las filas correctas sin revisar los 1000 libros.
imprimir_resultados(
    "CONSULTA 2 RÁPIDA: 3 Libros de la Categoría 1 (Usando JOIN)",
    """
    SELECT l.titulo, c.nombre AS categoria
    FROM libros l
    JOIN categorias c ON l.categoria_id = c.id
    WHERE l.categoria_id = 1
    LIMIT 3;
    """
)

# Esta es tu consulta "Jefe Final". Une 3 tablas usando la tabla puente.
# Gracias a los índices, encontrar a este autor y cruzar todos sus libros es instantáneo.
imprimir_resultados(
    "CONSULTA 3 RÁPIDA: Mejores libros de Lucas Fleitas (Cruce de 3 tablas)",
    """
    SELECT l.titulo, l.precio, l.rating, a.nombre AS autor
    FROM libros l
    JOIN libros_autores la ON l.id = la.libro_id
    JOIN autores a ON a.id = la.autor_id
    WHERE a.nombre = 'Lucas Fleitas'
    ORDER BY l.rating DESC
    LIMIT 5;
    """
)

# 4. CERRAMOS TODO
cursor.close()
conexion.close()
print("\n¡Desconexión exitosa. El análisis ha concluido!")


CONSULTA 1 LENTA: Top 5 libros más baratos
--------------------------------------------------
('An Abundance of Katherines', Decimal('10.00'))
('The Origin of Species', Decimal('10.01'))
('The Tipping Point: How Little Things Can Make a Big Difference', Decimal('10.02'))
('Patience', Decimal('10.16'))
('Greek Mythic History', Decimal('10.23'))

[⏳ Creando Índices en la base de datos...]
[✅ Índices creados exitosamente]

CONSULTA 2 RÁPIDA: 3 Libros de la Categoría 1 (Usando JOIN)
--------------------------------------------------
('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)', 'Sequential Art')
('Pop Gun War, Volume 1: Gift', 'Sequential Art')
('Patience', 'Sequential Art')

CONSULTA 3 RÁPIDA: Mejores libros de Lucas Fleitas (Cruce de 3 tablas)
--------------------------------------------------
('Private Paris (Private #10)', Decimal('47.61'), 5, 'Lucas Fleitas')
('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.', Decimal('23.11'